<a href="https://colab.research.google.com/github/Na-bra/recommendation-agent/blob/main/Recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install surprise

In [1]:
!pip3 install scikit-surprise
!pip3 install --force-reinstall -v "numpy<2.0.0"

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/3a/be/650f9c091ef71cb01d735775d554e068752d3ff63d7943b26316dc401749/numpy-1.21.2.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/5f/d6/ad58ded26556eaeaa8c971e08b6466f17c4ac4d786cd3d800e26ce59cc01/numpy-1.21.3.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/fb/48/b0708ebd7718a8933f0d3937513ef8ef2f4f04529f1f66ca86d873043921/numpy-1.21.4.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/c2/a8/a924a09492bdfee8c2ec3094d0

Imports

In [1]:
from surprise import Dataset,Reader
from surprise.prediction_algorithms import SVD
from surprise import accuracy
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

Load Data and Preprocess

In [2]:
path: str = "/content/drive/MyDrive/movie-data"
ratings_df = pd.read_csv(f"{path}/ratings.csv")
movies_df = pd.read_csv(f"{path}/movies.csv")

df = pd.merge(ratings_df,movies_df[['movieId', 'genres']], on = 'movieId', how = 'left')

Clean Data

In [13]:

df = pd.merge(ratings_df, movies_df[['movieId', 'genres']], on = 'movieId', how = 'left')

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()
mlb = MultiLabelBinarizer()
df['userId'] = user_encoder.fit_transform(df['userId'])
df['movieId'] = movie_encoder.fit_transform(df['movieId'])
df = df.join(pd.DataFrame(mlb.fit_transform(df.pop('genres').str.split('|')),columns = mlb.classes_, index = df.index))

In [14]:
df.drop(columns = "(no genres listed)", inplace = True)
df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,4.0,964982703,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,2,4.0,964981247,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,0,5,4.0,964982224,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,0,43,5.0,964983815,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,0,46,5.0,964982931,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100831,609,9416,4.0,1493848402,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
100832,609,9443,5.0,1493850091,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
100833,609,9444,5.0,1494273047,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
100834,609,9445,5.0,1493846352,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


Build The Model With Collaborative Filtering

In [15]:
train_df, test_df = train_test_split(df, test_size = 0.2)
train_df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
22572,152,8358,0.5,1525550704,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
922,6,2016,2.5,1106713072,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
70960,452,1347,4.0,972622981,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
87069,560,8671,3.5,1491091319,1,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
32970,224,92,5.0,949111497,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11860,72,5881,4.0,1464198972,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
13050,82,6029,3.0,1333843466,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
43603,291,1703,2.5,1483193888,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2359,18,371,3.0,965712068,0,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,0


In [16]:
reader = Reader(rating_scale = (0.5, 5.0))
data = Dataset.load_from_df(train_df[['userId','movieId','rating']], reader)
trainset = data.build_full_trainset()
trainset

In [17]:
model_svd = SVD()
model_svd.fit(trainset)
predictions_svd = model_svd.test(trainset.build_anti_testset())
accuracy.rmse(predictions_svd)

RMSE: 0.4769


0.4769084291346825

Make Recommendations

### Recommend Movies Based on a Liked Movie

### Demo: Get recommendations based on a liked movie

In [34]:
import re
from sklearn.metrics.pairwise import cosine_similarity

def get_top_n_reconmmendations(movie_title, n=5):
  # Convert input title to lowercase and remove year for robust comparison
  cleaned_input_title = movie_title.lower().strip()
  cleaned_input_title = re.sub(r'\s*\(\d{4}\)', '', cleaned_input_title)

  # Create a temporary series of titles from movies_df, removing years and converting to lowercase
  movies_df_cleaned_titles = movies_df['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.lower().str.strip()

  # Find the original movieId for the input movie using the cleaned titles
  matching_movies_indices = movies_df_cleaned_titles[movies_df_cleaned_titles == cleaned_input_title].index

  if matching_movies_indices.empty:
      print(f"Error: Movie '{movie_title}' not found in the database. Please check the spelling.")
      return []

  # Get the original movieId from the actual movies_df using the found index
  original_movie_id = movies_df.loc[matching_movies_indices[0], 'movieId']

  # Encode the original movieId using the pre-fitted movie_encoder
  try:
      encoded_movie_id = movie_encoder.transform([original_movie_id])[0]
  except ValueError:
      print(f"Error: Movie ID {original_movie_id} could not be encoded. It might not have been in the training data.")
      return []

  # Get the latent factor vector for the input movie from the SVD model
  if encoded_movie_id >= len(model_svd.qi):
      print(f"Error: Encoded movie ID {encoded_movie_id} is out of bounds for the SVD item factors.")
      return []

  target_movie_latent_factor = model_svd.qi[encoded_movie_id].reshape(1, -1)
  similarities = cosine_similarity(target_movie_latent_factor, model_svd.qi)
  top_similar_movie_indices = similarities.argsort()[0][::-1][1:n+1]


  similar_encoded_movie_ids = top_similar_movie_indices.tolist()
  similar_original_movie_ids = movie_encoder.inverse_transform(similar_encoded_movie_ids)
  recommended_titles = movies_df[movies_df['movieId'].isin(similar_original_movie_ids)]['title'].tolist()

  return recommended_titles

In [37]:
liked_movie_title = "Fantastic four"
recommendations = get_top_n_reconmmendations(liked_movie_title)

if recommendations:
  print(f"Top five recommended movies if you liked '{liked_movie_title}':")
  for i, title in enumerate(recommendations, start = 1):
    print(f"{i}. {title}")
else:
  print(f"No recommendations found for '{liked_movie_title}'.")

Top five recommended movies if you liked 'Fantastic four':
1. Star Trek: The Motion Picture (1979)
2. Big Fat Liar (2002)
3. Impromptu (1991)
4. Big Empty, The (2003)
5. Aristocrats, The (2005)
